# Session 13 — Pose Estimation

**Computer Vision (CVI4IC) · Summer Semester 2026 · FH Upper Austria**

This notebook is **practical-heavy** — most of the lecture's ideas come alive here:

1. **Three off-the-shelf models** side by side: MediaPipe BlazePose, YOLO11-pose, ViTPose.
2. **Decode raw heatmaps** yourself to see how keypoints come out of a network.
3. **Joint angles** + a tiny standing/sitting classifier.
4. **🕺 Mini project — Pose Karaoke:** upload any short video, watch the skeleton dance, and get a *dance signature*.
5. (Bonus) **Peek inside ViTPose features** — PCA → RGB and DINO-style patch similarity.
6. **Exercises** at the end.

> **Colab tip:** *Runtime → Change runtime type → T4 GPU*. CPU works too — ViTPose just becomes slow.


## 0 · Setup

Three libraries: `mediapipe`, `ultralytics`, `transformers` (for ViTPose). Plus the usual.

In [ ]:
!pip install -q -U mediapipe ultralytics transformers
# torch + torchvision are preinstalled on Colab.

In [ ]:
import os, math, json, time, urllib.request
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "·", "GPU" if DEVICE == "cuda" else "CPU only")
print("OpenCV:", cv2.__version__)

### Grab a few demo images

Free public-domain photos for testing. We'll use a yoga pose, a runner, and a multi-person scene.

In [ ]:
DEMO_URLS = {
    "yoga.jpg":   "https://upload.wikimedia.org/wikipedia/commons/thumb/c/c2/Mr-yoga-pose-warrior-3.jpg/640px-Mr-yoga-pose-warrior-3.jpg",
    "runner.jpg": "https://upload.wikimedia.org/wikipedia/commons/thumb/f/fe/Marathon_runner.jpg/640px-Marathon_runner.jpg",
    "crowd.jpg":  "https://upload.wikimedia.org/wikipedia/commons/thumb/a/ad/Group_of_People_Walking_on_Sidewalk.jpg/640px-Group_of_People_Walking_on_Sidewalk.jpg",
}
IMGS = Path("imgs"); IMGS.mkdir(exist_ok=True)
for name, url in DEMO_URLS.items():
    p = IMGS / name
    if not p.exists():
        try:
            urllib.request.urlretrieve(url, p)
        except Exception as e:
            print(f"!! Could not fetch {name}: {e}")
print("Files:", sorted(p.name for p in IMGS.glob("*")))

# Fallback: if anything failed, generate a synthetic person crop so the rest still works
if not (IMGS / "yoga.jpg").exists():
    img = np.full((480, 320, 3), 220, dtype=np.uint8)
    cv2.circle(img, (160, 80), 35, (200, 170, 140), -1)
    cv2.rectangle(img, (140, 115), (180, 280), (60, 100, 200), -1)
    cv2.rectangle(img, (110, 130), (140, 220), (60, 100, 200), -1)
    cv2.rectangle(img, (180, 130), (210, 220), (60, 100, 200), -1)
    cv2.rectangle(img, (145, 280), (165, 430), (40, 40, 40), -1)
    cv2.rectangle(img, (165, 280), (185, 430), (40, 40, 40), -1)
    cv2.imwrite(str(IMGS / "yoga.jpg"), img)
    print("Generated synthetic fallback for yoga.jpg")

In [ ]:
# Show the demo images
files = sorted(IMGS.glob("*.jpg"))
fig, axes = plt.subplots(1, len(files), figsize=(4 * len(files), 4))
if len(files) == 1: axes = [axes]
for ax, f in zip(axes, files):
    im = cv2.cvtColor(cv2.imread(str(f)), cv2.COLOR_BGR2RGB)
    ax.imshow(im); ax.axis("off"); ax.set_title(f.name)
plt.tight_layout(); plt.show()

---
## 1 · MediaPipe BlazePose — the easiest

5 lines. Runs on CPU, ~30 fps on a phone. **33 keypoints** (more than COCO's 17 — adds extra face / hand joints).

In [ ]:
import mediapipe as mp

mp_pose = mp.solutions.pose
mp_draw = mp.solutions.drawing_utils

def run_blazepose(img_bgr):
    """Returns (rgb_with_skeleton, list of (x, y) keypoints in pixel coords) or (img, None)."""
    with mp_pose.Pose(static_image_mode=True, model_complexity=1) as pose:
        rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        res = pose.process(rgb)
    H, W = img_bgr.shape[:2]
    if not res.pose_landmarks:
        return rgb, None
    kps = [(int(lm.x * W), int(lm.y * H)) for lm in res.pose_landmarks.landmark]
    annotated = rgb.copy()
    mp_draw.draw_landmarks(annotated, res.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                            mp_draw.DrawingSpec(color=(0, 200, 0), thickness=3, circle_radius=4),
                            mp_draw.DrawingSpec(color=(255, 130, 0), thickness=2))
    return annotated, kps

img = cv2.imread(str(IMGS / "yoga.jpg"))
ann, kps_bp = run_blazepose(img)
print(f"BlazePose found {len(kps_bp) if kps_bp else 0} keypoints")
plt.figure(figsize=(6, 7))
plt.imshow(ann); plt.title("MediaPipe BlazePose"); plt.axis("off")
plt.tight_layout(); plt.show()

---
## 2 · YOLO11-pose — single-stage, deployable

One network, one forward pass: persons + keypoints together. **17 COCO keypoints** per detected person.

In [ ]:
from ultralytics import YOLO

yolo_pose = YOLO("yolo11n-pose.pt")     # nano — fast; bump to s/m/l/x for accuracy

def run_yolo_pose(img_bgr):
    res = yolo_pose(img_bgr, imgsz=640, conf=0.3, verbose=False)[0]
    return res                              # has .plot(), .keypoints, .boxes

img = cv2.imread(str(IMGS / "yoga.jpg"))
res = run_yolo_pose(img)
ann = cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)
print(f"YOLO11-pose found {len(res.boxes)} person(s)")
plt.figure(figsize=(6, 7))
plt.imshow(ann); plt.title("YOLO11n-pose"); plt.axis("off")
plt.tight_layout(); plt.show()

---
## 3 · ViTPose — our deep dive

The plain-ViT model from the lecture. HuggingFace ships a clean `ViTPoseForPoseEstimation` wrapper, plus a `PersonImage` detector for stage 1.

> ViTPose is **top-down two-stage** — we need person boxes first. We use HuggingFace's recommended **RT-DETRv2** detector.

In [ ]:
from transformers import (
    AutoProcessor, RTDetrForObjectDetection,
    VitPoseImageProcessor, VitPoseForPoseEstimation,
)
from PIL import Image

# ---- Stage 1: person detector ----
det_id  = "PekingU/rtdetr_v2_r18vd"
det_proc = AutoProcessor.from_pretrained(det_id)
det_model = RTDetrForObjectDetection.from_pretrained(det_id).to(DEVICE).eval()

# ---- Stage 2: keypoint network ----
pose_id = "usyd-community/vitpose-base-simple"
pose_proc  = VitPoseImageProcessor.from_pretrained(pose_id)
pose_model = VitPoseForPoseEstimation.from_pretrained(pose_id).to(DEVICE).eval()
print("ViTPose stack ready ·", pose_id)

In [ ]:
def detect_persons(pil_img, conf=0.5):
    """Run RT-DETR, return boxes (xyxy pixel) of class=person only."""
    inputs = det_proc(images=pil_img, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = det_model(**inputs)
    target_size = torch.tensor([pil_img.size[::-1]]).to(DEVICE)
    results = det_proc.post_process_object_detection(outputs, threshold=conf,
                                                       target_sizes=target_size)[0]
    person_boxes = []
    for box, label in zip(results["boxes"], results["labels"]):
        # COCO class id 0 = person
        if det_model.config.id2label[label.item()] == "person":
            person_boxes.append(box.cpu().numpy())
    return np.array(person_boxes) if person_boxes else np.zeros((0, 4))


def run_vitpose(pil_img):
    """Returns: list of (boxes_xyxy, keypoints_array (17, 3)) per detected person."""
    boxes = detect_persons(pil_img)
    if len(boxes) == 0:
        return []
    # ViTPose expects boxes in xywh
    boxes_xywh = np.stack([boxes[:, 0], boxes[:, 1],
                            boxes[:, 2] - boxes[:, 0],
                            boxes[:, 3] - boxes[:, 1]], axis=1)
    inputs = pose_proc(pil_img, boxes=[boxes_xywh], return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = pose_model(**inputs)
    pose_results = pose_proc.post_process_pose_estimation(outputs, boxes=[boxes_xywh])
    return [(box, p) for box, p in zip(boxes, pose_results[0])]


pil = Image.open(IMGS / "yoga.jpg").convert("RGB")
results = run_vitpose(pil)
print(f"ViTPose found {len(results)} person(s)")

In [ ]:
# Visualize ViTPose result
fig, ax = plt.subplots(figsize=(6, 7))
ax.imshow(pil)
COCO_SKEL = [(15,13),(13,11),(16,14),(14,12),(11,12),(5,11),(6,12),(5,6),
             (5,7),(7,9),(6,8),(8,10),(1,2),(0,1),(0,2),(1,3),(2,4),(0,5),(0,6)]
for box, person in results:
    x1, y1, x2, y2 = box
    ax.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="yellow", lw=2))
    kp = person["keypoints"].cpu().numpy()           # (17, 2)
    sc = person["scores"].cpu().numpy()              # (17,)
    for a, b in COCO_SKEL:
        if sc[a] > 0.3 and sc[b] > 0.3:
            ax.plot([kp[a, 0], kp[b, 0]], [kp[a, 1], kp[b, 1]], "-", color="cyan", lw=2.5)
    for i, (x, y) in enumerate(kp):
        if sc[i] > 0.3:
            ax.plot(x, y, "o", markersize=7, mfc="orange", mec="black", mew=1.0)
ax.axis("off"); ax.set_title("ViTPose-base"); plt.tight_layout(); plt.show()

---
## 4 · Three models, same image

Quick visual comparison — pay attention to **occluded joints**, **fingers**, and **multi-person** behaviour.

In [ ]:
def annotate_yolo(img_bgr):
    res = yolo_pose(img_bgr, imgsz=640, conf=0.3, verbose=False)[0]
    return cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB), len(res.boxes)

def annotate_vitpose(pil_img):
    """Return RGB array with skeletons drawn."""
    arr = np.array(pil_img)
    results = run_vitpose(pil_img)
    for box, person in results:
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(arr, (x1, y1), (x2, y2), (255, 255, 0), 2)
        kp = person["keypoints"].cpu().numpy()
        sc = person["scores"].cpu().numpy()
        for a, b in COCO_SKEL:
            if sc[a] > 0.3 and sc[b] > 0.3:
                cv2.line(arr, tuple(map(int, kp[a])), tuple(map(int, kp[b])),
                         (0, 200, 255), 2)
        for i, (x, y) in enumerate(kp):
            if sc[i] > 0.3:
                cv2.circle(arr, (int(x), int(y)), 4, (255, 165, 0), -1)
    return arr, len(results)

# Run all three on the yoga image
img_bgr = cv2.imread(str(IMGS / "yoga.jpg"))
pil     = Image.open(IMGS / "yoga.jpg").convert("RGB")

bp_ann,    _ = run_blazepose(img_bgr)
yolo_ann,  n_yolo = annotate_yolo(img_bgr)
vit_ann,   n_vit  = annotate_vitpose(pil)

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
for ax, im, t in zip(axes, [bp_ann, yolo_ann, vit_ann],
                      ["MediaPipe BlazePose (33 kp)",
                       f"YOLO11n-pose ({n_yolo} person)",
                       f"ViTPose-base ({n_vit} person)"]):
    ax.imshow(im); ax.set_title(t); ax.axis("off")
plt.suptitle("Same image, three models"); plt.tight_layout(); plt.show()

---
## 5 · Build it yourself — decode raw heatmaps

The lecture talked about heatmaps. Here we inspect them on a ViTPose output. Hook the model to grab the raw output, then do argmax + sub-pixel refinement by hand.

In [ ]:
# Re-run ViTPose but keep the raw heatmap output
pil = Image.open(IMGS / "yoga.jpg").convert("RGB")
boxes = detect_persons(pil)
if len(boxes) == 0:
    print("No person detected — try a different image"); raise SystemExit

# Take the first person and run only the keypoint head
boxes_xywh = np.array([[boxes[0, 0], boxes[0, 1],
                        boxes[0, 2] - boxes[0, 0],
                        boxes[0, 3] - boxes[0, 1]]])
inputs = pose_proc(pil, boxes=[boxes_xywh], return_tensors="pt").to(DEVICE)
with torch.no_grad():
    outputs = pose_model(**inputs, output_hidden_states=False)

heatmaps = outputs.heatmaps[0].cpu().numpy()    # (17, H, W) — typically 64×48
print("Heatmap stack shape:", heatmaps.shape)

# Show the first 6 heatmaps side by side
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
NAMES_17 = ["nose","l-eye","r-eye","l-ear","r-ear","l-sh","r-sh","l-el","r-el",
             "l-wr","r-wr","l-hip","r-hip","l-kn","r-kn","l-an","r-an"]
for ax, k in zip(axes.ravel(), [0, 5, 7, 9, 11, 13]):
    ax.imshow(heatmaps[k], cmap="hot"); ax.set_title(f"k={k}  {NAMES_17[k]}"); ax.axis("off")
    y, x = np.unravel_index(heatmaps[k].argmax(), heatmaps[k].shape)
    ax.plot(x, y, "c+", markersize=15, mew=2)
plt.suptitle("ViTPose raw heatmaps  ·  cyan + = argmax peak"); plt.tight_layout(); plt.show()

In [ ]:
# Decode by hand: argmax + sub-pixel refinement
def decode_heatmaps(hm, box_xywh):
    """hm: (K, H, W) numpy array.  box_xywh: (4,) pixel coords of crop box.
       Returns (K, 2) keypoints in original-image pixel coordinates."""
    K, H, W = hm.shape
    bx, by, bw, bh = box_xywh
    keypoints = np.zeros((K, 2))
    for i in range(K):
        # Argmax
        y, x = np.unravel_index(hm[i].argmax(), hm[i].shape)
        # Sub-pixel offset (Newell et al.): quarter-pixel from the sign of the derivative
        dx = 0.25 * np.sign(hm[i, y, min(x+1, W-1)] - hm[i, y, max(x-1, 0)])
        dy = 0.25 * np.sign(hm[i, min(y+1, H-1), x] - hm[i, max(y-1, 0), x])
        # Heatmap coords → crop pixels → original image pixels
        keypoints[i] = [bx + (x + dx) * bw / W,
                        by + (y + dy) * bh / H]
    return keypoints

manual_kp = decode_heatmaps(heatmaps, boxes_xywh[0])
print("First 3 decoded keypoints (x, y):")
print(manual_kp[:3])

# Compare to the model's own decoder output
arr = np.array(pil)
fig, ax = plt.subplots(figsize=(6, 7))
ax.imshow(arr)
for x, y in manual_kp:
    ax.plot(x, y, "o", mfc="cyan", mec="black", mew=1, markersize=10)
ax.set_title("Hand-decoded keypoints (argmax + ¼-pixel refinement)"); ax.axis("off")
plt.tight_layout(); plt.show()

---
## 6 · Joint angles — from keypoints to semantics

The dot-product gives angles between bones:

$$\theta = \arccos\!\left(\frac{\mathbf{v}_1 \cdot \mathbf{v}_2}{\|\mathbf{v}_1\|\,\|\mathbf{v}_2\|}\right)$$

Common ones in fitness / physio:
- **Elbow angle** — shoulder–elbow–wrist
- **Knee angle** — hip–knee–ankle
- **Hip angle** — shoulder–hip–knee

In [ ]:
def joint_angle(p1, p2, p3):
    """Angle at p2 (in degrees) formed by p1–p2–p3."""
    v1 = np.array(p1) - np.array(p2)
    v2 = np.array(p3) - np.array(p2)
    cos = np.clip(v1.dot(v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-8), -1, 1)
    return float(np.degrees(np.arccos(cos)))

# Get the first person's keypoints
kp = results[0][1]["keypoints"].cpu().numpy()
angles = {
    "left elbow":   joint_angle(kp[5],  kp[7],  kp[9]),
    "right elbow":  joint_angle(kp[6],  kp[8],  kp[10]),
    "left knee":    joint_angle(kp[11], kp[13], kp[15]),
    "right knee":   joint_angle(kp[12], kp[14], kp[16]),
    "left hip":     joint_angle(kp[5],  kp[11], kp[13]),
    "right hip":    joint_angle(kp[6],  kp[12], kp[14]),
}
for name, a in angles.items():
    print(f"  {name:14s} {a:6.1f}°")

### A tiny posture classifier

Knee fully extended (>160°) → standing. Knee bent (<120°) → sitting / squatting. Try it on a few angles:

In [ ]:
def posture(kp):
    """Classify standing / sitting / something-else from leg + hip angles."""
    lk = joint_angle(kp[11], kp[13], kp[15])
    rk = joint_angle(kp[12], kp[14], kp[16])
    avg_knee = (lk + rk) / 2
    if avg_knee > 160: return f"standing  (knee = {avg_knee:.0f}°)"
    if avg_knee < 110: return f"sitting / squatting  (knee = {avg_knee:.0f}°)"
    return f"transition  (knee = {avg_knee:.0f}°)"

print(posture(kp))

---
## 7 · 🕺 Mini project — Pose Karaoke

The fun one. Upload **any short video** — a TikTok dance, a sports clip, you doing the floss, anything 5–30 seconds — and we:

1. Run **YOLO11-pose** on every frame (fast, deployable, no person-detection needed).
2. Produce two output videos:
   - **Overlay** — the original video with the skeleton drawn on top.
   - **Skeleton only** — black background, just the dancing skeleton.
3. Plot the **dance signature** — knee + elbow joint angles over time, your unique movement fingerprint.

> If you don't want to upload anything, the cell below synthesises a small "robot dance" video so the whole pipeline still runs end-to-end.


In [ ]:
# --- Helpers for video pose tracking ---
import pandas as pd
from base64 import b64encode
from IPython.display import HTML, display

# COCO skeleton edges + colours (re-used from §4)
COCO_SKEL = [(15,13),(13,11),(16,14),(14,12),(11,12),(5,11),(6,12),(5,6),
             (5,7),(7,9),(6,8),(8,10),(1,2),(0,1),(0,2),(1,3),(2,4),(0,5),(0,6)]

def draw_skeleton_cv2(canvas, kp, sc=None, thresh=0.3):
    """Draw COCO skeleton onto a BGR canvas in-place."""
    for a, b in COCO_SKEL:
        if sc is not None and (sc[a] < thresh or sc[b] < thresh):
            continue
        cv2.line(canvas,
                 (int(kp[a, 0]), int(kp[a, 1])),
                 (int(kp[b, 0]), int(kp[b, 1])),
                 (0, 230, 255), 3)
    for i, (x, y) in enumerate(kp):
        if sc is not None and sc[i] < thresh:
            continue
        cv2.circle(canvas, (int(x), int(y)), 4, (255, 165, 0), -1)
        cv2.circle(canvas, (int(x), int(y)), 4, (255, 255, 255), 1)

def show_video_inline(path, height=360):
    """Render a small MP4 inline via base64. Works in Colab + Jupyter."""
    data = b64encode(open(path, "rb").read()).decode()
    src  = f"data:video/mp4;base64,{data}"
    display(HTML(f'<video height="{height}" controls autoplay loop muted>'
                 f'<source src="{src}" type="video/mp4"></video>'))

def process_pose_video(in_path, out_path, mode="overlay", max_frames=300):
    """Run YOLO11-pose on every frame, write annotated video.

    Returns (n_frames_processed, angles_df).
    mode='overlay'        → skeleton on top of the original frame
    mode='skeleton_only'  → skeleton on a black background
    """
    cap = cv2.VideoCapture(in_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open {in_path}")
    fps  = cap.get(cv2.CAP_PROP_FPS) or 24
    W    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H    = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    vw   = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (W, H))

    angles = []
    n = 0
    while n < max_frames:
        ok, frame = cap.read()
        if not ok: break
        res = yolo_pose(frame, imgsz=640, conf=0.3, verbose=False)[0]

        if mode == "overlay":
            canvas = frame.copy()
        else:
            canvas = np.zeros_like(frame)

        # Draw the first detected person (focus on solo dancer)
        if res.keypoints is not None and len(res.keypoints):
            kp = res.keypoints.xy[0].cpu().numpy()
            sc = (res.keypoints.conf[0].cpu().numpy()
                  if res.keypoints.conf is not None else np.ones(17))
            draw_skeleton_cv2(canvas, kp, sc)
            # Record joint angles for the signature plot
            try:
                angles.append({
                    "frame":   n,
                    "l_knee":  joint_angle(kp[11], kp[13], kp[15]),
                    "r_knee":  joint_angle(kp[12], kp[14], kp[16]),
                    "l_elbow": joint_angle(kp[5],  kp[7],  kp[9]),
                    "r_elbow": joint_angle(kp[6],  kp[8],  kp[10]),
                })
            except Exception:
                pass

        vw.write(canvas)
        n += 1

    cap.release(); vw.release()
    return n, pd.DataFrame(angles)


In [ ]:
# --- Pick a video ---
# Try to use Colab's upload widget. Falls back to a synthetic robot-dance video
# if you skip the upload OR if you're not running in Colab.
SAMPLE = "robot_dance.mp4"
in_video = None

try:
    from google.colab import files
    print("Optional: upload a short video (≤30 s recommended). "
          "Press Cancel to use the built-in sample.")
    uploaded = files.upload()
    if uploaded:
        in_video = list(uploaded.keys())[0]
except Exception:
    pass

if in_video is None:
    # Synthesize a 4-second "robot dance" stick figure with arm waves + a squat
    def synth_dance(n_frames=120, fname=SAMPLE):
        Hh, Ww = 480, 320
        vw = cv2.VideoWriter(fname, cv2.VideoWriter_fourcc(*"mp4v"), 24, (Ww, Hh))
        for f in range(n_frames):
            t = f / n_frames
            wave = np.sin(2 * np.pi * t * 4)              # arms wave 4x
            squat = (np.cos(2 * np.pi * t * 2) + 1) / 2   # squat 2x
            img = np.full((Hh, Ww, 3), 235, dtype=np.uint8)
            crouch = squat * 70
            head_y = int(70 + crouch * 0.4)
            hip_y  = int(240 + crouch)
            knee_y = int(hip_y + 70 - crouch * 0.5)
            ank_y  = int(knee_y + 80 + crouch * 0.5)
            # head
            cv2.circle(img, (160, head_y), 26, (200, 170, 140), -1)
            # torso
            cv2.line(img, (160, head_y + 26), (160, hip_y), (60, 100, 200), 11)
            # arms (wave)
            arm_dx = int(60 + 20 * wave)
            arm_dy = int(-30 + 30 * wave)
            cv2.line(img, (160, head_y + 50),
                     (160 - arm_dx, head_y + 50 + arm_dy), (60, 100, 200), 8)
            cv2.line(img, (160, head_y + 50),
                     (160 + arm_dx, head_y + 50 - arm_dy), (60, 100, 200), 8)
            # legs (bend with squat)
            cv2.line(img, (148, hip_y),
                     (130 - int(crouch * 0.3), knee_y), (40, 40, 40), 10)
            cv2.line(img, (172, hip_y),
                     (190 + int(crouch * 0.3), knee_y), (40, 40, 40), 10)
            cv2.line(img, (130 - int(crouch * 0.3), knee_y),
                     (140, ank_y), (40, 40, 40), 10)
            cv2.line(img, (190 + int(crouch * 0.3), knee_y),
                     (180, ank_y), (40, 40, 40), 10)
            vw.write(img)
        vw.release()
    synth_dance()
    in_video = SAMPLE
    print(f"Using built-in sample: {SAMPLE}")
print("Input video:", in_video)


In [ ]:
# --- Run pose tracking on every frame ---
print("Processing overlay video (this can take ~20–60 s) …")
n_over, signature = process_pose_video(in_video, "with_skeleton.mp4", mode="overlay")
print(f"  overlay:        wrote {n_over} frames  →  with_skeleton.mp4")

print("Processing skeleton-only video …")
n_skel, _ = process_pose_video(in_video, "skeleton_only.mp4", mode="skeleton_only")
print(f"  skeleton-only:  wrote {n_skel} frames  →  skeleton_only.mp4")


In [ ]:
# --- Watch the results inline ---
display(HTML("<h4 style='margin-bottom:4px'>Overlay (original + skeleton)</h4>"))
show_video_inline("with_skeleton.mp4")

display(HTML("<h4 style='margin-top:18px;margin-bottom:4px'>Skeleton only 🕺</h4>"))
show_video_inline("skeleton_only.mp4")


In [ ]:
# --- Your dance signature: joint angles over time ---
if len(signature) == 0:
    print("No pose detected on any frame. Try a different video.")
else:
    fig, axes = plt.subplots(2, 1, figsize=(11, 5.5), sharex=True)
    for col, color in zip(["l_knee", "r_knee"], ["#3b82f6", "#ef4444"]):
        axes[0].plot(signature["frame"], signature[col], lw=1.8, color=color, label=col)
    for col, color in zip(["l_elbow", "r_elbow"], ["#3b82f6", "#ef4444"]):
        axes[1].plot(signature["frame"], signature[col], lw=1.8, color=color, label=col)
    axes[0].set_ylabel("Knee angle (°)")
    axes[1].set_ylabel("Elbow angle (°)"); axes[1].set_xlabel("Frame")
    for ax in axes:
        ax.legend(loc="upper right"); ax.grid(alpha=0.3)
        ax.set_ylim(40, 200)
    plt.suptitle("🕺 Your dance signature — joint angles over time", fontsize=13)
    plt.tight_layout(); plt.show()

    # Stats: how much you moved
    rng = signature[["l_knee", "r_knee", "l_elbow", "r_elbow"]].max()           - signature[["l_knee", "r_knee", "l_elbow", "r_elbow"]].min()
    print("\nRange of motion (°):"); print(rng.round(1).to_string())


### Want a *dance match*?

Run the same `process_pose_video` on **two** clips, store their signatures, then compare:

```python
_, sig_target = process_pose_video("reference.mp4", "ref_overlay.mp4")
_, sig_you    = process_pose_video("yours.mp4",     "you_overlay.mp4")

# Crop both to the same length, then correlation per joint
n = min(len(sig_target), len(sig_you))
corr = sig_target.iloc[:n].corrwith(sig_you.iloc[:n])
print("Per-joint Pearson correlation:")
print(corr.round(3))
print(f"\nDance match score: {corr.mean()*100:.1f} / 100")
```

A perfect copy gets ~100; flailing randomly gets near 0. Use this as the basis for the Pose Karaoke leaderboard 🏆.


---
## 8 · (Bonus) Peek inside ViTPose — feature maps + patch similarity

The output heatmaps tell us *where* each joint is. But the network is doing a lot more inside —
each of the 12 transformer blocks (ViTPose-base) produces a **(H/16 × W/16) grid of 768-dim feature tokens**.
Visualising those features tells us *how the representation builds up*.

We grab the **hidden states** of every block via `output_hidden_states=True`, then:

1.  **PCA → RGB** — project the 768-dim per-patch features down to 3 channels and visualise as colour. Same trick the DINOv2/v3 paper uses.
2.  **Patch similarity** — pick the patch under the nose, show its cosine similarity to every other patch (training-free segmentation, à la SAM teaser).

In [ ]:
# Re-run ViTPose, this time keep all 12 hidden states
inputs = pose_proc(pil, boxes=[boxes_xywh], return_tensors="pt").to(DEVICE)
with torch.no_grad():
    outputs = pose_model(**inputs, output_hidden_states=True)

# outputs.hidden_states is a tuple of (B, N, D) tensors — one per transformer block.
print(f"Hidden states: {len(outputs.hidden_states)} blocks")
print(f"Each block:    {outputs.hidden_states[0].shape}  (batch, n_tokens, dim)")
# Patch grid: ViTPose-base = 256x192 input / 16x16 patches = 16 tall × 12 wide = 192 tokens
N_TOK = outputs.hidden_states[0].shape[1]
patch_h, patch_w = 16, 12          # for 256×192 input with patch_size=16
assert patch_w * patch_h == N_TOK, f"Token count {N_TOK} doesn't match {patch_h}×{patch_w}"
print(f"Patch grid: {patch_h} × {patch_w}")

In [ ]:
# PCA → RGB visualisation for 3 blocks (early / middle / late)
from sklearn.decomposition import PCA

block_ids = [2, 6, 11]   # for ViTPose-base; use [4, 12, 23] for L, [6, 16, 31] for H
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
axes[0].imshow(pil); axes[0].set_title("Input crop"); axes[0].axis("off")

for ax, b in zip(axes[1:], block_ids):
    feats = outputs.hidden_states[b][0].cpu().numpy()      # (N, D)
    pca = PCA(n_components=3)
    rgb = pca.fit_transform(feats)                          # (N, 3)
    # Normalise per channel to [0, 1]
    rgb = (rgb - rgb.min(0)) / (rgb.max(0) - rgb.min(0) + 1e-8)
    rgb = rgb.reshape(patch_h, patch_w, 3)
    ax.imshow(rgb, interpolation="nearest")
    ax.set_title(f"Block {b}  (PCA → RGB)")
    ax.axis("off")
plt.suptitle("ViTPose internal features — same scene, three depths", fontsize=12)
plt.tight_layout(); plt.show()

**What to see:**
- **Early block (≈2)** — features cluster by *colour and texture*. Sky-blue, skin-tone, shadow.
- **Middle block (≈6)** — features cluster by *body part region*. Torso, arms, legs become coherent blobs.
- **Late block (≈11)** — features become *keypoint-aware*. The decoder can now read off heatmap peaks.

This is the classic "early = pixels, late = semantics" picture from any pretrained ViT — connects to what Christoph showed for DINOv3 in session 11/12.

In [ ]:
# Patch similarity — DINO-style. Pick a query token (the patch closest to the nose)
# and show cosine similarity with every other patch.
feats = outputs.hidden_states[-1][0]                       # (N, D), last block
feats = feats / feats.norm(dim=-1, keepdim=True)           # L2 normalise

# Use the model's own nose prediction (kp 0) projected to patch grid
crop_pred = run_vitpose(pil)[0][1]["keypoints"][0].cpu().numpy()  # nose (x, y) in original image
bx, by, bw, bh = boxes_xywh[0]
nose_xc = (crop_pred[0] - bx) / bw * 192      # 256×192 crop, x in 192 px
nose_yc = (crop_pred[1] - by) / bh * 256
nose_patch = int(nose_yc / 16) * patch_w + int(nose_xc / 16)
nose_patch = max(0, min(N_TOK - 1, nose_patch))
print(f"Nose maps to patch {nose_patch}  /  {N_TOK}")

# Cosine sim of nose-patch with every patch
sim = (feats @ feats[nose_patch]).cpu().numpy()
sim_grid = sim.reshape(patch_h, patch_w)

# Resize similarity to the crop size for overlay
from scipy.ndimage import zoom
sim_up = zoom(sim_grid, (256/patch_h, 192/patch_w), order=1)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(pil); axes[0].set_title("Input — cyan + = nose"); axes[0].axis("off")
axes[0].plot(crop_pred[0], crop_pred[1], "c+", markersize=20, mew=3)
axes[1].imshow(sim_up, cmap="inferno", vmin=0)
axes[1].set_title("Cosine similarity to the nose patch"); axes[1].axis("off")
plt.suptitle("DINO-style patch similarity — emergent face-region from features alone", fontsize=12)
plt.tight_layout(); plt.show()

---

## 9 · Going further — Sapiens2 (current SOTA, 2026)

ViTPose is a great teaching model. The current **state of the art** is **Sapiens2** (Meta, ICLR 2026) — a
direct successor with the same plain-ViT recipe pushed to its limits:

| Property             | ViTPose-base  | Sapiens2-1B         |
| -------------------- | ------------- | ------------------- |
| Input resolution     | 256 × 192     | **1024 × 768**      |
| Parameters           | 90 M          | 1.46 B              |
| Pretraining data     | ImageNet (1M) | **1B human images** |
| Tasks per backbone   | 1 (pose)      | **4** (pose + body-part seg + normals + pointmaps) |
| COCO val AP          | 75.8          | **82.3**            |
| Inference GPU memory | ~2 GB         | **~30 GB**          |

The catch is the last row: Sapiens2-1B doesn't fit on a free Colab T4. Want to try it anyway?
Quick start (will OOM on T4 — try Colab Pro / A100):

```python
# !pip install -q safetensors
# from safetensors.torch import load_file
# from sapiens.backbones.standalone.sapiens2 import Sapiens2
#
# model = Sapiens2(arch="sapiens2_0.4b", img_size=(1024, 768), patch_size=16).eval().cuda()
# ckpt = "sapiens2_0.4b_pretrain.safetensors"     # download from HuggingFace
# model.load_state_dict(load_file(ckpt))
```

For our notebook ViTPose-base stays the right choice — the **ideas** are the same, the
**hardware bill** is 15× smaller. Sapiens2 inherits everything we just saw: plain ViT encoder,
heatmap output, top-down two-stage pipeline. Just bigger.

---

## ✏️ Exercises

**1 · Edge cases.** Run all three models (BlazePose, YOLO11-pose, ViTPose) on `crowd.jpg` (multi-person scene). Which model handles multiple people best? Which one misses the occluded persons?

**2 · Pose comparison — yoga match.** Take two yoga photos (or two of yourself, same pose). Extract keypoints, normalise each by the bounding-box diagonal, then compute the **mean Euclidean distance** between the two keypoint sets. Lower = better match.

**3 · 🕺 Pose Karaoke — your turn.** Record a 10–20 second video of yourself doing **any** repeatable motion (squats, jumping jacks, the floss, even just waving). Run §7 and look at your dance signature — can you spot every rep in the joint-angle plot?

**4 · Dance match leaderboard.** Use the *Want a dance match?* recipe at the end of §7 to compare two clips: a reference dance and someone trying to copy it. Whose copy scores highest? **Tip:** normalise the signatures (subtract mean, divide by std) before correlating, so people of different builds compete fairly.

**5 · Multi-person karaoke.** Modify `process_pose_video` to draw **all** detected persons, not just the first. Re-run on a clip with two people. Bonus: assign each person a different skeleton colour.

**6 · Feature-map exploration (advanced).** §8 extracts ViT features and shows PCA / patch similarity. Pick a non-nose keypoint (e.g. *left wrist*) and repeat the patch-similarity heatmap. Does the network's notion of "wrist-ness" generalise — does it also light up the *right* wrist?

Your code:


In [ ]:
# Your solution here
